# ScanOps 보안 모델 — 클라우드 GPU(Colab) 학습

M3 MacBook Air에서 ~2시간 걸리는 QLoRA 학습을, Colab 무료 **T4 GPU**에서 진짜 4bit QLoRA로 **약 5~10분**에 끝냅니다.

**사전 준비:** 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택.

흐름: ① 의존성 설치 → ② 학습 데이터 업로드 → ③ QLoRA 학습 → ④ 어댑터 다운로드 → (로컬에서 GGUF 변환·서빙)

## ① 의존성 설치 + GPU 확인

In [ ]:
!pip -q install transformers peft datasets accelerate bitsandbytes
import torch
assert torch.cuda.is_available(), 'GPU가 없습니다. 런타임 유형을 T4 GPU로 바꾸세요.'
print('GPU:', torch.cuda.get_device_name(0))

## ② 학습 데이터 업로드
로컬 `data/lora_train_v7.jsonl`(또는 최신 버전)을 업로드합니다. 실행 후 파일 선택 창에서 jsonl을 고르세요.

In [ ]:
from google.colab import files
up = files.upload()                 # lora_train_v7.jsonl 선택
TRAIN_FILE = list(up.keys())[0]
print('업로드:', TRAIN_FILE)

## ③ QLoRA 학습 (진짜 4bit, CUDA)
로컬 `ml/train.py`와 동일한 방법(Qwen2.5-Coder-1.5B + LoRA r=32 + cross-entropy + AdamW/cosine)을 self-contained로 담았습니다.

In [ ]:
import json, time, torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          DataCollatorForSeq2Seq, Trainer, TrainingArguments)

BASE = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
MAXLEN, R, ALPHA, EPOCHS, LR = 1024, 32, 64, 3, 1e-4

rows = [json.loads(l) for l in open(TRAIN_FILE) if l.strip()]
n_none = sum(1 for r in rows if r['completion'].split(chr(10))[0].upper().startswith('VULNERABILITY: NONE'))
print(f'{len(rows)}개 예시, 안전(NONE) {100*n_none/len(rows):.1f}%')
ds = Dataset.from_list(rows).train_test_split(test_size=0.1, seed=42)

tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token

def fmt(e):
    return {'text': '<|im_start|>system\nYou are a security code analyzer.<|im_end|>\n'
            '<|im_start|>user\n'+e['prompt']+'<|im_end|>\n'
            '<|im_start|>assistant\n'+e['completion']+'<|im_end|>'}
def tk(e):
    o = tok(e['text'], truncation=True, max_length=MAXLEN); o['labels'] = o['input_ids'].copy(); return o
tr = ds['train'].map(fmt).map(tk, remove_columns=ds['train'].column_names+['text'])
ev = ds['test'].map(fmt).map(tk, remove_columns=ds['test'].column_names+['text'])

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=R, lora_alpha=ALPHA, lora_dropout=0.05,
        target_modules=['q_proj','k_proj','v_proj','o_proj'], task_type=TaskType.CAUSAL_LM))
model.print_trainable_parameters()

args = TrainingArguments(output_dir='out', num_train_epochs=EPOCHS, per_device_train_batch_size=4,
        gradient_accumulation_steps=2, learning_rate=LR, lr_scheduler_type='cosine', warmup_ratio=0.03,
        fp16=True, logging_steps=10, eval_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True, report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=ev,
        data_collator=DataCollatorForSeq2Seq(tok, pad_to_multiple_of=8, return_tensors='pt'))
t0=time.time(); trainer.train(); print(f'학습 {(time.time()-t0)/60:.1f}분')

model.save_pretrained('adapter'); tok.save_pretrained('adapter')
json.dump([{'step':e['step'],'loss':e.get('loss'),'eval_loss':e.get('eval_loss')}
           for e in trainer.state.log_history if 'loss' in e or 'eval_loss' in e],
          open('adapter/train_loss.json','w'), indent=2)

## ④ 어댑터 다운로드
내려받은 `adapter.zip`을 로컬 `models/qwen-security-qlora-v8/`에 풀고, `scripts/convert_to_gguf_v5.py`(태그만 v8로)로 GGUF 변환·Ollama 등록 후 `python -m ml.evaluate`로 평가하세요.

In [ ]:
!cd adapter && zip -r ../adapter.zip . >/dev/null
from google.colab import files
files.download('adapter.zip')